<a href="https://colab.research.google.com/github/anaulbrich12/project-based-learning/blob/master/projeto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
#Instala a biblioteca do Gemini
!pip install -q -U google-generativeai

from google.colab import userdata
import google.generativeai as genai

#Importacao da chave Colab e configuracao do modelo
api_key = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=api_key)

model = genai.GenerativeModel('gemini-1.5-flash')
print("Gemini configurado e pronto! :)")



Gemini configurado e pronto! :)


In [14]:
!pip install -q python-docx

import os
import docx

def carregar_base_de_conhecimento(pasta="documentos"):
    base_texto = []

    # Lista e ordena os arquivos para garantir a leitura sequencial
    arquivos = sorted([f for f in os.listdir(pasta) if f.endswith('.docx')])

    for arquivo in arquivos:
        caminho = os.path.join(pasta, arquivo)
        doc = docx.Document(caminho)

        texto_doc = []

        # 1. Extrai texto dos parágrafos
        for p in doc.paragraphs:
            if p.text.strip():
                texto_doc.append(p.text.strip())

        # 2. Extrai texto das tabelas
        for table in doc.tables:
            for row in table.rows:
                linha = " | ".join([cell.text.strip().replace('\n', ' ') for cell in row.cells])
                if linha.strip():
                    texto_doc.append(linha)

        # Consolida o arquivo atual
        conteudo_formatado = f"=== DOCUMENTO: {arquivo} ===\n" + "\n".join(texto_doc)
        base_texto.append(conteudo_formatado)

    return "\n\n" + ("="*50) + "\n\n".join(base_texto)

# Carrega todos os documentos da pasta 'documentos'
contexto_corporativo = carregar_base_de_conhecimento("documentos")
print(f"✅ Base de conhecimento carregada com sucesso! Total de caracteres: {len(contexto_corporativo)}")

✅ Base de conhecimento carregada com sucesso! Total de caracteres: 16772


In [15]:
from google.colab import drive
import os

# Conecta o pasta documentos (Google Drive) ao Colab
drive.mount('/content/drive')

# Caminho da pasta dentro do Google Drive
caminho_drive = '/content/drive/MyDrive/documentos'

if os.path.exists(caminho_drive):
    print("✅ Google Drive conectado e pasta 'documentos' encontrada com sucesso!")
else:
    print("⚠️ Conectado, mas certifique-se de que criou a pasta 'documentos' na raiz do seu Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive conectado e pasta 'documentos' encontrada com sucesso!


In [16]:
# Passa o caminho do Google Drive para a função
contexto_corporativo = carregar_base_de_conhecimento('/content/drive/MyDrive/documentos')
print(f"✅ Base de conhecimento carregada do Drive! Total de caracteres: {len(contexto_corporativo)}")

✅ Base de conhecimento carregada do Drive! Total de caracteres: 16772


In [17]:
# Instruções de comportamento, persona e regras do Agente ExCambio
system_instruction = f"""
Você é o Agente de IA Corporativo da ExCambio Corretora.
Sua função é responder dúvidas de colaboradores internos (Atendimento, Operações, Compliance e Jurídico) de forma precisa e embasada unicamente nas políticas oficiais da empresa.

DIRETRIZES DE ATENDIMENTO:
1. Responda estritamente com base nos documentos corporativos fornecidos.
2. Seja direto, didático e profissional.
3. Mencione sempre a ÁREA RESPONSÁVEL (Owner) competente quando o assunto envolver escalonamento ou aprovação formal.
4. Para Câmbio Empresarial (PJ), aplique rigorosamente as alíquotas (IOF 3,5%, IRRF 15% ou 25% para paraísos fiscais) e mencione o código NBS/Fato Gerador quando relevante.
5. Se a dúvida não puder ser respondida com base no contexto, informe educadamente que a informação não consta na base corporativa e oriente o colaborador a abrir um chamado com a área responsável.
6. Mencione o nome do documento a qual buscou a informação como fonte.

=== BASE DE CONHECIMENTO CORPORATIVA (EXCAMBIO) ===
{contexto_corporativo}
"""

# Inicializa o modelo Gemini com as instruções e documentos embarcados
agente_excambio = genai.GenerativeModel(
    model_name='gemini-1.5-flash',
    system_instruction=system_instruction
)

# Inicia a sessão de chat
chat = agente_excambio.start_chat()
print("🤖 Agente de IA ExCambio configurado e pronto para responder!")

🤖 Agente de IA ExCambio configurado e pronto para responder!


In [ ]:
# Instala e lança a interface do colaborador
!pip install -q gradio
import gradio as gr

def responder_colaborador(mensagem, historico):
    resposta = chat.send_message(mensagem)
    return resposta.text

demo = gr.ChatInterface(
    fn=responder_colaborador,
    title="🏦 Agente de IA Corporativo — ExCambio Corretora",
    description="Portal interno de suporte ao colaborador para consultas de câmbio, compliance, enquadramento fiscal (NBS/O Fato) e limites operacionais.",
    examples=[
        "Qual é a alíquota de IOF para remessa de Pessoa Jurídica?",
        "Uma empresa quer pagar consultoria no exterior (NBS 1.0101.10.00). Qual a alíquota de IRRF e quem analisa 'O Fato'?",
        "O cliente quer cancelar uma operação fechada há 20 minutos, é possível?",
        "Qual o limite operacional para cliente PF Prime e quando exige dupla checagem?"
    ]
)

# Gera o link público para acesso externo
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://33ec1174b73589228a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
